# GCP Compute Engine 인스턴스 생성 및 Cloud Ops Agent 설정

이 노트북은 Google Cloud Platform(GCP)에서 Compute Engine VM 인스턴스를 프로비저닝하고, Cloud Ops Agent 정책을 생성 및 적용하는 스크립트를 포함하고 있습니다.

### 환경 사전 확인
- Google Cloud SDK(gcloud)가 설치되어 있어야 합니다.
- gcloud auth login 및 프로젝트 권한이 설정되어 있어야 합니다.
- 타겟 프로젝트: iceu-songpa10
- 타겟 리전/존: us-central1-a

## 1. Compute Engine VM 인스턴스 생성
- 인스턴스명: instance-20260915-143200
- 머신 유형: e2-medium
- 부팅 디스크: Debian 13 (Trixie), 10GB pd-balanced
- 레이블: goog-ops-agent-policy=v2-template-1-7-0, goog-ec-src=vm_add-gcloud

In [ ]:
import datetime

# 인스턴스 기본 설정
PROJECT_ID = "iceu-songpa10"
ZONE = "us-central1-a"
# 타임스탬프 기반 고유 인스턴스명 자동 생성 (중복 충돌 방지)
INSTANCE_NAME = f"instance-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"생성 대상 인스턴스명: {INSTANCE_NAME} (Zone: {ZONE})")

!gcloud compute instances create {INSTANCE_NAME} \
    --project={PROJECT_ID} \
    --zone={ZONE} \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=920380215419-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name={INSTANCE_NAME},disk-resource-policy=projects/{PROJECT_ID}/regions/us-central1/resourcePolicies/default-schedule-1,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any


## 2. Cloud Ops Agent 설정 파일 (config.yaml) 생성
주피터 노트북 환경(Windows, Linux, macOS)에 구애받지 않도록 %%writefile 매직 명령어를 사용하여 config.yaml을 생성합니다.

In [7]:
%%writefile config.yaml
agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0


Overwriting config.yaml


## 3. Cloud Ops Agent 정책 생성 및 상태 확인
위에서 생성한 `config.yaml` 설정을 기반으로 Ops Agent 정책을 생성 및 적용합니다.

> **안내**: 이미 정책이 존재하는 경우 `ALREADY_EXISTS` 에러를 방지하도록 상태를 자동 확인하며, 재성성을 원하는 경우 삭제 명령어 안내를 제공합니다.

In [8]:
import sys
import subprocess
import shutil

# Windows 콘솔 인코딩 안전 처리
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

POLICY_ID = "goog-ops-agent-v2-template-1-7-0-us-central1-a"
PROJECT_ID = "iceu-songpa10"
ZONE = "us-central1-a"

gcloud_bin = shutil.which('gcloud.cmd') or shutil.which('gcloud') or 'gcloud'

# 1. 정책이 이미 존재하는지 OS Policy Assignment 확인 (ALREADY_EXISTS 에러 방지)
check_cmd = [
    gcloud_bin, "compute", "os-config", "os-policy-assignments", "describe",
    POLICY_ID, f"--location={ZONE}", f"--project={PROJECT_ID}", "--format=value(rolloutState)"
]
check_proc = subprocess.run(check_cmd, capture_output=True, text=True, encoding='utf-8', errors='ignore')

if check_proc.returncode == 0:
    state = check_proc.stdout.strip()
    print(f"[확인] Cloud Ops Agent 정책 [{POLICY_ID}]이(가) 이미 존재하며 활성화 상태입니다. (Rollout State: {state})")
    print("   - 적용 대상 라벨: goog-ops-agent-policy=v2-template-1-7-0")
    print("\n[안내] 정책을 완전히 삭제하고 다시 생성하려면 아래 주석을 해제하고 실행하세요:")
    print(f"# !gcloud compute os-config os-policy-assignments delete {POLICY_ID} --location={ZONE} --project={PROJECT_ID} --quiet")
else:
    print(f"[진행] Cloud Ops Agent 정책 [{POLICY_ID}] 생성을 시작합니다...")
    !gcloud compute instances ops-agents policies create {POLICY_ID} \
        --project={PROJECT_ID} \
        --zone={ZONE} \
        --file=config.yaml


[확인] Cloud Ops Agent 정책 [goog-ops-agent-v2-template-1-7-0-us-central1-a]이(가) 이미 존재하며 활성화 상태입니다. (Rollout State: SUCCEEDED)
   - 적용 대상 라벨: goog-ops-agent-policy=v2-template-1-7-0

[안내] 정책을 완전히 삭제하고 다시 생성하려면 아래 주석을 해제하고 실행하세요:
# !gcloud compute os-config os-policy-assignments delete goog-ops-agent-v2-template-1-7-0-us-central1-a --location=us-central1-a --project=iceu-songpa10 --quiet


## 4. 인스턴스 생성 결과 확인

In [9]:
!gcloud compute instances list --project=iceu-songpa10

NAME                      ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
instance-20260914-054845  us-central1-a  e2-medium                  10.128.0.3   35.226.19.68  RUNNING


---
### (참고) 원본 Bash 스크립트
Linux / macOS / Google Cloud Shell 등 Bash 셸 환경에서 한 번에 실행할 때 사용할 수 있는 원본 스크립트입니다. (Windows 주피터 실행 방해 방지를 위해 참고용 마크다운으로 제공됩니다.)


```bash
# 1. 인스턴스 생성
gcloud compute instances create instance-20260914-054845 \
    --project=iceu-songpa10 \
    --zone=us-central1-a \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=920380215419-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-054845,disk-resource-policy=projects/iceu-songpa10/regions/us-central1/resourcePolicies/default-schedule-1,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any \
&& \
printf 'agentsRule:\n  packageState: installed\n  version: latest\ninstanceFilter:\n  inclusionLabels:\n  - labels:\n      goog-ops-agent-policy: v2-template-1-7-0\n' > config.yaml \
&& \
gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a \
    --project=iceu-songpa10 \
    --zone=us-central1-a \
    --file=config.yaml
```


## 5. 미삭제 자원 및 야간 과금 방지 점검 (Resource Audit Harness)
인스턴스 생성/삭제 작업 후 제대로 삭제되지 않아 발생하는 익일 과금을 방지하기 위해 남아있는 리소스(VM, 미연결 디스크, 미사용 정적 IP 등)를 점검합니다.

In [11]:
import sys
# 현재 주피터 커널의 정확한 Python 인터프리터로 리소스 점검 스크립트 실행
!{sys.executable} .agents/skills/gcp-resource-audit/scripts/audit_resources.py


[GCP 리소스 과금 방지 점검 리포트] Project: iceu-songpa10

[안전] 현재 프로젝트에 남아있는 유료 인스턴스나 고아 리소스가 없습니다.
   - 실행 중인 VM: 0개
   - 미연결 디스크: 0개
   - 미사용 정적 IP: 0개
   * 오늘 밤 과금 위험 없이 안심하고 마감하셔도 됩니다.



## 6. 동일 사양 기준 최저가 리전 분석 및 비용 비교 (Top 3)

현재 생성된 인스턴스 옵션(`e2-medium`, 2 vCPU / 4 GB RAM, 10 GB `pd-balanced` 부팅 디스크)을 기준으로, 전 세계 주요 GCP 리전별 요금을 비교하여 **가장 저렴한 리전과 Top 3**를 분석합니다.

In [12]:
import os
import json
import pandas as pd

VM_TYPE = "e2-medium"        # 2 vCPU, 4GB RAM
DISK_TYPE = "pd-balanced"
DISK_SIZE_GB = 10
HOURS_PER_MONTH = 730         # GCP 표준 월 환산 시간 (365일 / 12달 * 24시간)

# 1. 데이터 소스 로드 (cheapest_regions_data.json이 있으면 32개 전체 리전 데이터 활용, 없으면 기본 데이터 활용)
json_path = 'cheapest_regions_data.json'
region_name_map = {
    "us-central1": "미국 아이오와 (Iowa)",
    "us-east1": "미국 사우스캐롤라이나 (S. Carolina)",
    "us-west1": "미국 오리건 (Oregon)",
    "us-east5": "미국 콜럼버스 (Columbus)",
    "us-west8": "미국 유타 (Utah)",
    "europe-north2": "유럽 스톡홀름 (Stockholm)",
    "northamerica-south1": "캐나다 캘거리 (Calgary)",
    "europe-west1": "유럽 벨기에 (Belgium)",
    "europe-north1": "유럽 핀란드 (Finland)",
    "us-east4": "미국 버지니아 (N. Virginia)",
    "asia-east1": "아시아 대만 (Taiwan)",
    "asia-northeast3": "아시아 서울 (Seoul)",
}

if os.path.exists(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    records = []
    for d in data:
        r = d['region']
        records.append({
            'region': r,
            'name': region_name_map.get(r, r),
            'vm_hourly': d['vm_hourly'],
            'vm_monthly': d['vm_monthly'],
            'disk_monthly': d['disk_monthly'],
            'total_monthly': d['total_monthly']
        })
else:
    records = [
        {"region": "us-central1", "name": "미국 아이오와 (Iowa)", "vm_hourly": 0.03351, "disk_gb_monthly": 0.10},
        {"region": "us-east1", "name": "미국 사우스캐롤라이나 (S. Carolina)", "vm_hourly": 0.03351, "disk_gb_monthly": 0.10},
        {"region": "us-west1", "name": "미국 오리건 (Oregon)", "vm_hourly": 0.03351, "disk_gb_monthly": 0.10},
        {"region": "us-east5", "name": "미국 콜럼버스 (Columbus)", "vm_hourly": 0.03351, "disk_gb_monthly": 0.10},
        {"region": "europe-north2", "name": "유럽 스톡홀름 (Stockholm)", "vm_hourly": 0.03518, "disk_gb_monthly": 0.10},
        {"region": "northamerica-south1", "name": "캐나다 캘거리 (Calgary)", "vm_hourly": 0.03652, "disk_gb_monthly": 0.109},
        {"region": "europe-west1", "name": "유럽 벨기에 (Belgium)", "vm_hourly": 0.03686, "disk_gb_monthly": 0.11},
        {"region": "europe-north1", "name": "유럽 핀란드 (Finland)", "vm_hourly": 0.03686, "disk_gb_monthly": 0.11},
        {"region": "us-east4", "name": "미국 버지니아 (N. Virginia)", "vm_hourly": 0.03774, "disk_gb_monthly": 0.11},
        {"region": "asia-east1", "name": "아시아 대만 (Taiwan)", "vm_hourly": 0.03787, "disk_gb_monthly": 0.11},
        {"region": "asia-northeast3", "name": "아시아 서울 (Seoul)", "vm_hourly": 0.04356, "disk_gb_monthly": 0.13},
    ]
    for item in records:
        item["vm_monthly"] = item["vm_hourly"] * HOURS_PER_MONTH
        item["disk_monthly"] = item["disk_gb_monthly"] * DISK_SIZE_GB
        item["total_monthly"] = item["vm_monthly"] + item["disk_monthly"]

records.sort(key=lambda x: x['total_monthly'])
min_cost = records[0]['total_monthly']

# 2. 판다스 데이터프레임 생성 (주피터에서 열 정렬 완벽 지원)
table_data = []
for idx, r in enumerate(records[:12], 1):
    diff = r['total_monthly'] - min_cost
    diff_str = f"(+${diff:.2f})" if diff > 0.001 else "[최저가]"
    table_data.append({
        '순위': idx,
        '리전 (Region)': r['region'],
        '위치': r['name'],
        '시간당(VM)': f"${r['vm_hourly']:.5f}",
        '월간(VM)': f"${r['vm_monthly']:.2f}",
        '월간(디스크)': f"${r['disk_monthly']:.2f}",
        '총 예상 월비용': f"${r['total_monthly']:.2f}",
        '비고': diff_str
    })

df = pd.DataFrame(table_data)
try:
    from IPython.display import display
    display(df)
except Exception:
    print(df.to_string(index=False))

# 3. 요약 리포트 출력
print(f"\n[최저가 Top 3 리전 요약]")
for i, r in enumerate(records[:3], 1):
    print(f"{i}위: {r['region']} ({r['name']}) -> 총 ${r['total_monthly']:.2f}/월 (VM: ${r['vm_hourly']:.5f}/h, 디스크: ${r['disk_monthly']:.2f}/월)")

# 서울 리전(asia-northeast3) 비용 차이 정확한 계산
seoul_item = next((item for item in records if item['region'] == 'asia-northeast3'), None)
if seoul_item:
    seoul_saving = seoul_item['total_monthly'] - min_cost
    print(f"\n참고: 현재 사용 중인 us-central1 (아이오와)은 월 ${min_cost:.2f}로 이미 글로벌 최저가 그룹(1위)에 속해 있습니다.")
    print(f"      (서울 리전 ${seoul_item['total_monthly']:.2f}/월 대비 매월 약 ${seoul_saving:.2f} 비용 절감)")


,순위,리전 (Region),위치,시간당(VM),월간(VM),월간(디스크),총 예상 월비용,비고
0,1,us-west8,미국 유타 (Utah),$0.03351,$24.46,$1.00,$25.46,[최저가]
1,2,us-east5,미국 콜럼버스 (Columbus),$0.03351,$24.46,$1.00,$25.46,[최저가]
2,3,europe-north2,유럽 스톡홀름 (Stockholm),$0.03518,$25.68,$1.00,$26.68,(+$1.22)
3,4,northamerica-south1,캐나다 캘거리 (Calgary),$0.03652,$26.66,$1.09,$27.75,(+$2.29)
4,5,africa-south1,africa-south1,$0.03686,$26.91,$1.10,$28.00,(+$2.54)
5,6,me-west1,me-west1,$0.03686,$26.91,$1.10,$28.01,(+$2.55)
6,7,europe-west4,europe-west4,$0.03689,$26.93,$1.10,$28.03,(+$2.57)
7,8,europe-north1,유럽 핀란드 (Finland),$0.03689,$26.93,$1.10,$28.03,(+$2.57)
8,9,us-east4,미국 버지니아 (N. Virginia),$0.03774,$27.55,$1.10,$28.65,(+$3.19)
9,10,us-west4,us-west4,$0.03774,$27.55,$1.10,$28.65,(+$3.19)



[최저가 Top 3 리전 요약]
1위: us-west8 (미국 유타 (Utah)) -> 총 $25.46/월 (VM: $0.03351/h, 디스크: $1.00/월)
2위: us-east5 (미국 콜럼버스 (Columbus)) -> 총 $25.46/월 (VM: $0.03351/h, 디스크: $1.00/월)
3위: europe-north2 (유럽 스톡홀름 (Stockholm)) -> 총 $26.68/월 (VM: $0.03518/h, 디스크: $1.00/월)

참고: 현재 사용 중인 us-central1 (아이오와)은 월 $25.46로 이미 글로벌 최저가 그룹(1위)에 속해 있습니다.
      (서울 리전 $32.68/월 대비 매월 약 $7.22 비용 절감)


## 7. 최저가 리전으로 신규 인스턴스 프로비저닝 코드
분석된 최저가 리전(`us-central1`, `us-east1`, `us-west1` 등)을 선택하여 동일한 옵션으로 VM을 즉시 생성할 수 있는 스크립트입니다.

In [ ]:
# 최저가 리전 선택 (1위: us-central1, us-east1, us-west1 중 선택)
TARGET_PROJECT = "iceu-songpa10"
SELECTED_ZONE = "us-central1-a"  # 또는 "us-east1-b", "us-west1-a"
INSTANCE_NAME = "instance-cheapest-tier1"

create_cmd = (
    f"gcloud compute instances create {INSTANCE_NAME} "
    f"--project={TARGET_PROJECT} "
    f"--zone={SELECTED_ZONE} "
    f"--machine-type=e2-medium "
    f"--network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default "
    f"--metadata=enable-osconfig=TRUE "
    f"--maintenance-policy=MIGRATE "
    f"--provisioning-model=STANDARD "
    f"--service-account=920380215419-compute@developer.gserviceaccount.com "
    f"--scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append "
    f"--create-disk=auto-delete=yes,boot=yes,device-name={INSTANCE_NAME},image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced "
    f"--no-shielded-secure-boot "
    f"--shielded-vtpm "
    f"--shielded-integrity-monitoring "
    f"--labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud "
    f"--reservation-affinity=any"
)

print("생성할 gcloud 명령어:")
print(create_cmd)

# 실제 생성을 실행하려면 주피터에서 아래 코드의 주석을 해제하고 실행하세요:
# import subprocess
# subprocess.run(create_cmd, shell=True, check=True)


## 8. Gemini 챗봇 서비스 배포 및 HTTPS(보안 연결) 자동 구성

생성된 Compute Engine 인스턴스에 Gemini 챗봇 웹 애플리케이션을 배포하고, **Nginx 리버스 프록시** 및 **Let's Encrypt(`sslip.io`)**를 연동하여 안전한 **HTTPS 보안 연결**을 구성합니다.

### 주요 구성 항목
1. **Secret Manager IAM 연동**: `roles/secretmanager.secretAccessor` 권한 바인딩
2. **인바운드 방화벽 포트 개방**: `allow-chatbot-port` (TCP 5000, 80, 443)
3. **Nginx 리버스 프록시 및 포트 라우팅**: 80(HTTP->HTTPS 301 리다이렉트), 443(HTTPS), 5000(HTTPS 호환)
4. **Let's Encrypt 공인 인증서**: `sslip.io` 와일드카드 DNS를 통한 공인 CA 인증서 자동 발급 및 갱신

In [ ]:
import subprocess
import shutil

PROJECT_ID = "iceu-songpa10"
ZONE = "us-central1-a"
FIREWALL_RULE = "allow-chatbot-port"

gcloud_bin = shutil.which('gcloud.cmd') or shutil.which('gcloud') or 'gcloud'

print("=== [1] 방화벽 포트 (5000, 80, 443) 구성 확인 ===")
fw_check = subprocess.run([gcloud_bin, "compute", "firewall-rules", "describe", FIREWALL_RULE, f"--project={PROJECT_ID}"],
                          capture_output=True, text=True, encoding='utf-8', errors='ignore')
if fw_check.returncode != 0:
    print(f"-> 방화벽 규칙 [{FIREWALL_RULE}] 생성 중...")
    !{gcloud_bin} compute firewall-rules create {FIREWALL_RULE} --allow=tcp:5000,tcp:80,tcp:443 --target-tags=chatbot-server --project={PROJECT_ID}
else:
    print(f"-> 방화벽 규칙 [{FIREWALL_RULE}]이 이미 존재합니다.")

print("\n=== [2] 챗봇 자동 배포 및 HTTPS 구성 스크립트 실행 안내 ===")
print("1) 인스턴스 배포: python deploy_to_gcp.py")
print("2) HTTPS 구성: bash setup_https.sh")
print("3) 엔드포인트 검증: python verify_https.py")


## 9. GCP 자원 완벽 반납 및 익일 과금 방지 종합 정리 가이드 (Comprehensive Teardown & Cleanup)

인스턴스를 삭제하더라도 사용자가 모르는 사이에 백그라운드에 남아 계속 과금을 유발하는 **숨은 자원들을 모두 안전하게 점검하고 일괄 반납/삭제**하는 통합 절차입니다.

### ⚠️ 사용자가 놓치기 쉬운 과금 유발 잔여 리소스 7가지
1. **실행 중인 Compute VM 인스턴스**: 초당/시간당 vCPU 및 RAM 요금 발생
2. **미연결 영구 디스크 (Unattached Disks)**: VM을 삭제해도 부팅 디스크의 `auto-delete=no`였거나 추가 디스크는 남아서 GB당 월간 디스크 요금 계속 청구
3. **미사용 고정 공인 IP (Unused Static External IPs)**: VM에 바인딩되지 않은 고정 IP는 **보유하고 있는 것만으로 시간당 유휴 요금 발생**
4. **사용자 지정 방화벽 포트 규칙 (Firewall Rules)**: 개방된 불필요한 포트(5000, 80, 443 등) 잔존 시 보안 취약점 노출
5. **부하 분산기 및 포워딩 규칙 (Forwarding Rules / Target Pools)**: 생성된 로드밸런서 리소스 잔존 시 시간당 과금
6. **스냅샷 및 스케줄 정책 (Snapshots & Resource Policies)**: 테스트 시 생성된 스냅샷 용량 요금
7. **Cloud Ops Agent OS Policy Assignment**: VM 삭제 후 불필요하게 남아있는 OS 정책 할당 정리

In [ ]:
import json
import subprocess
import shutil

PROJECT_ID = "iceu-songpa10"
ZONE = "us-central1-a"
POLICY_ID = "goog-ops-agent-v2-template-1-7-0-us-central1-a"
gcloud_bin = shutil.which('gcloud.cmd') or shutil.which('gcloud') or 'gcloud'

def run_gcloud(args):
    cmd = [gcloud_bin] + args + [f"--project={PROJECT_ID}", "--format=json"]
    res = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8', errors='ignore')
    if res.returncode == 0 and res.stdout.strip():
        try:
            return json.loads(res.stdout)
        except Exception:
            return []
    return []

def audit_and_cleanup_all(dry_run=True):
    print("=" * 65)
    mode_str = "[점검 모드 (DRY-RUN)]" if dry_run else "[실제 삭제 및 반납 진행 모드 (ACTION)]"
    print(f"{mode_str} GCP 전체 자원 반납 상태 점검: {PROJECT_ID}")
    print("=" * 65)
    
    # 1. Compute Instances
    instances = run_gcloud(["compute", "instances", "list"])
    print(f"\n[1/7] Compute Engine VM 인스턴스: {len(instances)}개 발견")
    for inst in instances:
        name, z = inst.get('name'), inst.get('zone', '').split('/')[-1]
        print(f"   - 발견: {name} (Zone: {z}, 상태: {inst.get('status')})")
        if not dry_run:
            print(f"     -> 인스턴스 삭제 실행 중: {name}...")
            subprocess.run([gcloud_bin, "compute", "instances", "delete", name, f"--zone={z}", f"--project={PROJECT_ID}", "--quiet"])
            
    # 2. Disks (미연결 디스크 집중 점검)
    disks = run_gcloud(["compute", "disks", "list"])
    unattached_disks = [d for d in disks if not d.get('users')]
    print(f"\n[2/7] 영구 디스크: 전체 {len(disks)}개 중 미연결 고아 디스크 {len(unattached_disks)}개 발견")
    for d in unattached_disks:
        dname, dz = d.get('name'), d.get('zone', '').split('/')[-1]
        print(f"   - 미연결 디스크: {dname} ({d.get('sizeGb')}GB, Zone: {dz}) -> 과금 유발 대상!")
        if not dry_run:
            print(f"     -> 고아 디스크 삭제 실행 중: {dname}...")
            subprocess.run([gcloud_bin, "compute", "disks", "delete", dname, f"--zone={dz}", f"--project={PROJECT_ID}", "--quiet"])
            
    # 3. Static IP Addresses (미사용 정적 IP 점검)
    addresses = run_gcloud(["compute", "addresses", "list"])
    unused_ips = [a for a in addresses if a.get('status') == 'RESERVED']
    print(f"\n[3/7] 공인 IP 주소: 전체 {len(addresses)}개 중 미사용 유휴 고정 IP {len(unused_ips)}개 발견")
    for ip in unused_ips:
        ip_name, r = ip.get('name'), ip.get('region', '').split('/')[-1]
        print(f"   - 미사용 고정 IP: {ip_name} ({ip.get('address')}) -> 시간당 유휴 요금 발생 중!")
        if not dry_run:
            print(f"     -> 고정 IP 반납/삭제 실행 중: {ip_name}...")
            subprocess.run([gcloud_bin, "compute", "addresses", "delete", ip_name, f"--region={r}", f"--project={PROJECT_ID}", "--quiet"])
            
    # 4. Custom Firewall Rules
    fws = run_gcloud(["compute", "firewall-rules", "list"])
    custom_fws = [f for f in fws if not f.get('name', '').startswith('default-')]
    print(f"\n[4/7] 사용자 지정 방화벽 포트 규칙: {len(custom_fws)}개 발견")
    for fw in custom_fws:
        fw_name = fw.get('name')
        print(f"   - 커스텀 방화벽: {fw_name} (허용 포트: {fw.get('allowed')})")
        if not dry_run:
            print(f"     -> 방화벽 포트 반납/삭제 실행 중: {fw_name}...")
            subprocess.run([gcloud_bin, "compute", "firewall-rules", "delete", fw_name, f"--project={PROJECT_ID}", "--quiet"])
            
    # 5. Forwarding Rules & Target Pools
    frs = run_gcloud(["compute", "forwarding-rules", "list"])
    tps = run_gcloud(["compute", "target-pools", "list"])
    print(f"\n[5/7] 로드밸런싱/포워딩 규칙: {len(frs)}개, 타겟 풀: {len(tps)}개")
    if not dry_run:
        for fr in frs:
            subprocess.run([gcloud_bin, "compute", "forwarding-rules", "delete", fr.get('name'), f"--project={PROJECT_ID}", "--quiet"])
        for tp in tps:
            subprocess.run([gcloud_bin, "compute", "target-pools", "delete", tp.get('name'), f"--project={PROJECT_ID}", "--quiet"])
            
    # 6. Snapshots
    snaps = run_gcloud(["compute", "snapshots", "list"])
    print(f"\n[6/7] 스냅샷: {len(snaps)}개 발견")
    if not dry_run:
        for s in snaps:
            subprocess.run([gcloud_bin, "compute", "snapshots", "delete", s.get('name'), f"--project={PROJECT_ID}", "--quiet"])
            
    # 7. Cloud Ops Agent OS Policy
    print(f"\n[7/7] Cloud Ops Agent 정책 [{POLICY_ID}] 점검...")
    if not dry_run:
        subprocess.run([gcloud_bin, "compute", "os-config", "os-policy-assignments", "delete", POLICY_ID, f"--location={ZONE}", f"--project={PROJECT_ID}", "--quiet"])
        print(f"     -> OS 정책 삭제 명령 전송 완료")
        
    print("\n" + "=" * 65)
    if dry_run:
        print("[안내] 위 항목 중 삭제가 필요한 자원이 있다면 audit_and_cleanup_all(dry_run=False)를 호출하여 일괄 반납할 수 있습니다.")
    else:
        print("[완료] 모든 지정 자원의 반납 및 삭제 조치가 완료되었습니다.")
    print("=" * 65)

# 1. 현재 잔여 자원 현황 점검 (기본값: 안전한 Dry-run 모드)
audit_and_cleanup_all(dry_run=True)

# 2. 일괄 삭제 및 완전 반납을 실행하려면 아래 주석을 해제하고 실행하세요:
# audit_and_cleanup_all(dry_run=False)
